# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kobeyvines/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Signal 1 — CTR vs. position tier (flag-linked).** This is the signal behind FlyRank's `low_ctr_visible_page` flag. Position determines how much CTR a page can realistically earn, so comparing a page's CTR to the *global* average is misleading — a position-9 page with 2% CTR might be completely normal, and a position-2 page with 2% CTR is a real problem. I'm using the same position-tier framing from w03: pages are bucketed into position tiers (NTILE(5) equivalent on `avg_position`), and CTR is compared against the *median CTR of that page's own tier*, not the dataset-wide median.

**Signal 2 — Staleness vs. impressions.** This is the signal behind `stale_visible_page`. The assumption a refresh rule leans on is: old pages that still carry real visibility are worth touching; old pages nobody sees are not. I check whether `days_since_last_update` buckets actually correspond to different visibility levels — if stale pages have collapsed to near-zero impressions, staleness alone isn't a useful trigger and the rule needs a volume floor riding alongside it regardless.

**The rule, in plain words:** A page is worth reviewing for metadata/title work if its CTR sits meaningfully below what pages *at the same position tier* typically earn, and it has enough impression volume that the gap can't be noise.

**Reason code:** `low_ctr_for_position` — CTR below this page's own position-tier median, at sufficient volume.

**Action label:** `review_metadata` — title/meta/snippet review, not a content rewrite. This rule only measures a click-through gap at a given position; it says nothing about whether the *content* itself is weak.

**Assumption flagged for review:** volume floor is set to `impressions_90d >= 500` to match the starter pipeline's own `low_ctr_visible_page` threshold (section 5 of the lane guide). Section 2's code prints how many rows this floor captures — adjust `MIN_IMPRESSIONS` below if that count looks too strict or too loose for a top-10/top-20 review.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Load ---
# Starter dataset per the lane guide (section 5), resolved from the current directory
# or any parent so the notebook works from the repo root or its workspace parent.
DATA_RELATIVE_PATH = Path("data/raw/content_refresh_anonymized.csv")
DATA_PATH = next(
    (base / DATA_RELATIVE_PATH for base in [Path.cwd(), *Path.cwd().parents]
     if (base / DATA_RELATIVE_PATH).is_file()),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATA_RELATIVE_PATH} from the current directory or its parents. "
        f"Current directory: {Path.cwd()}"
    )
df = pd.read_csv(DATA_PATH)

# --- Same inclusion rule the starter feature prep uses (lane guide section 5) ---
# impressions_90d > 0 and content_age_days >= 90, deduped by content_id.
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset="content_id")
print(f"Rows after inclusion filter: {len(df):,}")

# --- Position tiers (NTILE(5) equivalent) ---
# Only rows with a real position can be tiered.
has_position = df["avg_position"] > 0
df["position_tier"] = pd.Series(index=df.index, dtype="object")
df.loc[has_position, "position_tier"] = pd.qcut(
    df.loc[has_position, "avg_position"], 5, labels=[1, 2, 3, 4, 5]
)

# ============================================================
# Signal check #1 — CTR vs. position tier (flag-linked)
# ============================================================
signal1 = df.loc[has_position].groupby("position_tier", observed=True).agg(
    n=("ctr", "size"),
    median_ctr=("ctr", "median"),
    mean_ctr=("ctr", "mean"),
).reset_index()
print("\nSignal 1 — CTR by position tier (1 = best position, 5 = worst)")
print(signal1.to_string(index=False))

# Verdict logic: CONFIRMED if median CTR clearly decreases as tier worsens (monotonic-ish),
# so comparing a page to its OWN tier is meaningfully different from comparing it to the
# global median. If tiers show little separation, the tier adjustment isn't earning its keep.
ctr_by_tier = signal1.set_index("position_tier")["median_ctr"]
is_monotonic_decreasing = ctr_by_tier.is_monotonic_decreasing
print(f"\nVerdict — Signal 1: "
      f"{'CONFIRMED' if is_monotonic_decreasing else 'MIXED'} "
      f"(median CTR {'decreases' if is_monotonic_decreasing else 'does not cleanly decrease'} "
      f"as position tier worsens; n={len(df.loc[has_position]):,})")

# ============================================================
# Signal check #2 — Staleness vs. impressions (flag-linked)
# ============================================================
staleness_bins = [-1, 90, 180, np.inf]
staleness_labels = ["<90d", "90-180d", "180d+"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"], bins=staleness_bins, labels=staleness_labels
)
signal2 = df.groupby("staleness_bucket", observed=True).agg(
    n=("impressions_90d", "size"),
    median_impressions=("impressions_90d", "median"),
    mean_impressions=("impressions_90d", "mean"),
).reset_index()
print("\nSignal 2 — Impressions by staleness bucket")
print(signal2.to_string(index=False))

# Verdict logic: CONFIRMED if the 180d+ bucket still carries real median impressions
# (i.e., stale pages are still visible, which is the assumption stale_visible_page relies on).
# FALSE if stale pages have effectively collapsed to near-zero visibility — in that case,
# staleness alone would flag pages nobody will ever see anyway.
stale_row = signal2[signal2["staleness_bucket"] == "180d+"]
stale_median = stale_row["median_impressions"].iloc[0] if len(stale_row) else np.nan
fresh_median = signal2[signal2["staleness_bucket"] == "<90d"]["median_impressions"].iloc[0]
verdict2 = "CONFIRMED" if stale_median >= 0.5 * fresh_median else "MIXED"
print(f"\nVerdict — Signal 2: {verdict2} "
      f"(180d+ median impressions {stale_median:.0f} vs. <90d median {fresh_median:.0f}; "
      f"n={len(df):,})")

# NOTE: fill in the verdict words above by hand once you've actually run this against
# real data — the thresholds here (monotonic check, 0.5x ratio) are a starting heuristic,
# not a rule to trust blindly. Read the printed tables yourself before you accept them.

Rows after inclusion filter: 30,000

Signal 1 — CTR by position tier (1 = best position, 5 = worst)
 position_tier    n  median_ctr  mean_ctr
             1 5942       0.180  1.273847
             2 5595       0.130  0.462595
             3 5802       0.110  0.367304
             4 5708       0.085  0.315787
             5 5748       0.000  0.151816

Verdict — Signal 1: CONFIRMED (median CTR decreases as position tier worsens; n=28,795)

Signal 2 — Impressions by staleness bucket
staleness_bucket     n  median_impressions  mean_impressions
            <90d 20655               472.0       4219.161317
         90-180d  9171              1692.0       7486.665140
           180d+   174                15.5       1172.448276

Verdict — Signal 2: MIXED (180d+ median impressions 16 vs. <90d median 472; n=30,000)


## 2. Build the ranked queue (writes the CSV)

Score: a page is flagged if its CTR sits below its own position tier's median CTR **and** `impressions_90d >= MIN_IMPRESSIONS`. This is a boolean gate, not a weighted blend — the rule stays readable in one sentence, per the baseline skill's own bar.

One reason code (`low_ctr_for_position`), one action label (`review_metadata`). Score magnitude is the CTR gap size (tier median minus the page's own CTR), so ties within the flagged set still rank sensibly — bigger gap, higher priority.

In [7]:
MIN_IMPRESSIONS = 500  # see the assumption note in Section 1 — adjust after checking row counts below

eligible = df.loc[has_position].copy()
eligible["tier_median_ctr"] = eligible["position_tier"].map(ctr_by_tier)
eligible["ctr_gap"] = eligible["tier_median_ctr"] - eligible["ctr"]

flagged = eligible[
    (eligible["ctr_gap"] > 0) & (eligible["impressions_90d"] >= MIN_IMPRESSIONS)
].copy()

print(f"Rows eligible (have a position tier): {len(eligible):,}")
print(f"Rows flagged at MIN_IMPRESSIONS={MIN_IMPRESSIONS}: {len(flagged):,} "
      f"({len(flagged) / len(eligible):.1%} of eligible)")

# If this feels too strict/loose, that's the signal to change MIN_IMPRESSIONS above and re-run —
# not to silently widen the score condition.

flagged["reason_code"] = "low_ctr_for_position"
flagged["action"] = "review_metadata"
flagged["score"] = flagged["ctr_gap"]  # bigger gap = higher priority

queue_cols = [
    "content_id", "client_id", "position_tier", "avg_position",
    "ctr", "tier_median_ctr", "ctr_gap", "impressions_90d",
    "score", "reason_code", "action",
]
ranked_queue = flagged[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"\nWrote {len(ranked_queue):,} rows to work/outputs/baseline_action_score.csv")
ranked_queue.head(10)

Rows eligible (have a position tier): 28,795
Rows flagged at MIN_IMPRESSIONS=500: 4,312 (15.0% of eligible)

Wrote 4,312 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,position_tier,avg_position,ctr,tier_median_ctr,ctr_gap,impressions_90d,score,reason_code,action
0,content_04d69956e256,client_19581e27de,1,3.7,0.0,0.18,0.18,742,0.18,low_ctr_for_position,review_metadata
1,content_8f2559c3bc1b,client_6208ef0f77,1,5.7,0.0,0.18,0.18,740,0.18,low_ctr_for_position,review_metadata
2,content_9983d31c53cb,client_4e07408562,1,5.5,0.0,0.18,0.18,7737,0.18,low_ctr_for_position,review_metadata
3,content_05c60d0e3a6a,client_7f2253d7e2,1,5.9,0.0,0.18,0.18,1041,0.18,low_ctr_for_position,review_metadata
4,content_cbae09e4ff9b,client_349c41201b,1,5.8,0.0,0.18,0.18,792,0.18,low_ctr_for_position,review_metadata
5,content_e319408369bd,client_19581e27de,1,5.4,0.0,0.18,0.18,551,0.18,low_ctr_for_position,review_metadata
6,content_b854e1b19fe8,client_a88a7902cb,1,3.5,0.0,0.18,0.18,1194,0.18,low_ctr_for_position,review_metadata
7,content_37bb865c7f2f,client_f369cb89fc,1,5.7,0.0,0.18,0.18,1112,0.18,low_ctr_for_position,review_metadata
8,content_7e97f31ba9bb,client_19581e27de,1,0.7,0.0,0.18,0.18,603,0.18,low_ctr_for_position,review_metadata
9,content_e0b39e983a86,client_6208ef0f77,1,4.9,0.0,0.18,0.18,1434,0.18,low_ctr_for_position,review_metadata


## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong. This is where a rule that looked fine on paper gets checked against real rows — fill in the printed table by reading the actual content_ids your run produces (they'll differ from any example run).

In [8]:
top10 = ranked_queue.head(10).copy()
print(top10.to_string(index=False))

# For each row above, write one line covering:
#   - action: review_metadata (fixed for this rule)
#   - why: position tier + ctr_gap size + impressions_90d, in plain words
#     e.g. "Tier 2 page, CTR 1.1% vs. tier median 3.4%, 1,850 impressions — clear gap at real volume."
#   - what would make it wrong: e.g. intent mismatch (page ranks for a query type where low CTR is
#     expected regardless of metadata), recent consolidation (a sibling page absorbed the clicks —
#     see lane guide section 7), or a position that's borderline between two tiers.
#
# Do this by hand against your actual output — don't template it without looking at the rows.

          content_id         client_id position_tier  avg_position  ctr  tier_median_ctr  ctr_gap  impressions_90d  score          reason_code          action
content_04d69956e256 client_19581e27de             1           3.7  0.0             0.18     0.18              742   0.18 low_ctr_for_position review_metadata
content_8f2559c3bc1b client_6208ef0f77             1           5.7  0.0             0.18     0.18              740   0.18 low_ctr_for_position review_metadata
content_9983d31c53cb client_4e07408562             1           5.5  0.0             0.18     0.18             7737   0.18 low_ctr_for_position review_metadata
content_05c60d0e3a6a client_7f2253d7e2             1           5.9  0.0             0.18     0.18             1041   0.18 low_ctr_for_position review_metadata
content_cbae09e4ff9b client_349c41201b             1           5.8  0.0             0.18     0.18              792   0.18 low_ctr_for_position review_metadata
content_e319408369bd client_19581e27de        

## 4. Weak picks + leakage check

**Weak picks:** flag any of the top 10 above where the reason code technically fires but the underlying explanation looks like something other than a metadata problem — most likely a consolidation case (lane guide section 7: a sibling page absorbed the clicks a low-CTR page appears to be missing) or a position tier boundary artifact (a page barely inside a worse tier than it functionally belongs to).

**Leakage check — confirmed by inspection of the code above:**
- No `trend_direction` or any other product-computed flag used as an input to the score.
- No future-window fields — `ctr`, `avg_position`, `impressions_90d`, and `days_since_last_update` are all current-window observed signals, not forward-looking.
- `tier_median_ctr` is computed from the same current window, not a separate historical period — so it can't be smuggling in future information either.
- The rule does not use, rebuild, or approximate any of FlyRank's own scored outputs (`health_score`, `priority_score`, `action_type`) as an input.

In [9]:
# Optional: quick automated leakage sanity check — confirms the queue columns don't include
# any column name that looks like a product decision field or a trend/future label.
banned_substrings = ["health_score", "priority_score", "action_type", "trend_direction", "needs_", "is_quick_win"]
leaked = [c for c in ranked_queue.columns if any(b in c for b in banned_substrings)]
print("Leakage check:", "CLEAN — no flagged column names in the queue." if not leaked else f"REVIEW: {leaked}")

Leakage check: CLEAN — no flagged column names in the queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.